In [2]:
import sqlite3
import numpy as np
import pandas as pd

DB_PATH = "data/data.db"
conn = sqlite3.connect(DB_PATH)

raw = pd.read_sql_query("SELECT * FROM ohlc_hw", conn)
raw.head()

,ts,open,high,low,close,adj_close,volume,ticker
0,1975-04-14,5.4300,5.4800,5.3600,5.4400,1.640000,446392,CAT
1,1978-04-05,4.8400,4.8900,4.8100,4.8800,0.656990,355200,CVX
2,2018-05-25,124.5400,124.7600,121.2200,122.1900,92.220000,8936649,CVX
3,1995-12-07,9.9700,9.9700,9.8300,9.9400,5.880000,3954150,HD
4,1981-07-23,0.1038,0.1038,0.1038,0.1038,0.080013,34451232,AAPL


In [3]:
raw.info()
raw.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 397711 entries, 0 to 397710
Data columns (total 8 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   ts         397711 non-null  object 
 1   open       397711 non-null  float64
 2   high       397711 non-null  float64
 3   low        397711 non-null  float64
 4   close      397711 non-null  float64
 5   adj_close  377856 non-null  float64
 6   volume     397711 non-null  int64  
 7   ticker     397711 non-null  object 
dtypes: float64(5), int64(1), object(2)
memory usage: 24.3+ MB


,open,high,low,close,adj_close,volume
count,397711.000000,397711.000000,397711.000000,397711.000000,377856.000000,3.977110e+05
mean,46.285479,46.747660,45.816279,46.292903,51.652250,3.097773e+07
std,67.132594,67.778210,66.477178,67.141879,194.584045,1.266237e+08
min,0.000000,0.005208,0.004801,0.005208,0.002945,0.000000e+00
25%,4.209938,4.240000,4.160000,4.200000,1.390000,1.730700e+06
50%,19.850000,20.060000,19.630000,19.860000,11.960000,4.648200e+06
75%,57.530000,58.130000,56.950000,57.530000,43.770000,1.102380e+07
max,604.260000,608.630000,597.880000,605.400000,10442.510783,9.276606e+09


In [5]:
print("Date range:", raw["ts"].min(), "to", raw["ts"].max())
print("Number of distinct tickers:", raw["ticker"].nunique())
raw["ticker"].unique()[:40]

Date range: 1962-01-02 to 2024-12-06
Number of distinct tickers: 150


array(['CAT', 'CVX', 'HD', 'AAPL', 'TRV', 'JNJ', 'DIS',
       '<a target="_blank"><span style="color: blue">CSCO</span></a>',
       'SHW', 'KO',
       '<a target="_blank"><span style="font-weight: bold; color: blue">HD</span></a>',
       '<a target="_blank"><span style="font-weight: bold; color: blue">VZ</span></a>',
       'IBM', 'PG', 'HON', 'AXP', 'MMM', 'GS', 'BA', 'CRM', 'UNH', 'NKE',
       'NVDA', 'AMGN', 'CSCO', 'MCD',
       '<a target="_blank"><span style="color: red">AMGN</span></a>',
       'VZ', 'V',
       '<a target="_blank"><span style="color: red">DIS</span></a>',
       '<a target="_blank"><span style="font-weight: bold; color: blue">NKE</span></a>',
       '<a target="_blank"><span style="color: blue">TRV</span></a>',
       'MRK', 'WMT',
       '<a target="_blank"><span style="font-weight: bold; color: red">SHW</span></a>',
       '<a target="_blank"><span style="font-weight: bold; color: red">JNJ</span></a>',
       '<a target="_blank"><span style="font-weight:

In [14]:
def normalize_ticker(label: str) -> str:
    no_html = pd.Series([label]).str.replace(r"<[^>]+>", "", regex=True).iloc[0]
    return no_html.strip().upper()

df = raw.copy()
df["ticker_raw"] = df["ticker"]                      
df["ticker"] = df["ticker_raw"].map(normalize_ticker)

print("Distinct tickers after normalization:", df["ticker"].nunique())
print("Rows with HTML-wrapped labels:", (df["ticker_raw"] != df["ticker"]).sum())
sorted(df["ticker"].unique())

Distinct tickers after normalization: 30
Rows with HTML-wrapped labels: 39936


['AAPL',
 'AMGN',
 'AMZN',
 'AXP',
 'BA',
 'CAT',
 'CRM',
 'CSCO',
 'CVX',
 'DIS',
 'GS',
 'HD',
 'HON',
 'IBM',
 'JNJ',
 'JPM',
 'KO',
 'MCD',
 'MMM',
 'MRK',
 'MSFT',
 'NKE',
 'NVDA',
 'PG',
 'SHW',
 'TRV',
 'UNH',
 'V',
 'VZ',
 'WMT']

In [15]:

price_cols = ["open", "high", "low", "close", "adj_close", "volume"]
key = ["ticker", "ts"]

exact_dup_mask = df.duplicated(subset=key + price_cols, keep="first")
print("Exact duplicate rows:", exact_dup_mask.sum())
df = df[~exact_dup_mask].copy()

conflict_mask = df.duplicated(subset=key, keep=False)
print("Rows with conflicting (ticker, ts) keys:", conflict_mask.sum())
print("Rows after removing exact duplicates:", len(df))
df = df.sort_values(["ticker", "ts"]).reset_index(drop=True)
ref = pd.read_sql_query("SELECT ticker, ts, close, adj_close FROM ohlc", conn)

def find_missing_days(data, calendar):
    merged = calendar[["ticker", "ts"]].merge(data[["ticker", "ts"]], how="left", indicator=True)
    return merged[merged["_merge"] == "left_only"][["ticker", "ts"]]

missing_days = find_missing_days(df, ref)
print("Missing trading days:", len(missing_days))

for ticker in sorted(ref["ticker"].unique()):
    dates = ref[ref["ticker"] == ticker]["ts"]
    n = (missing_days["ticker"] == ticker).sum()
    print(f"{ticker} is missing {n} trading days between {dates.min()} and {dates.max()}")

Exact duplicate rows: 29460
Rows with conflicting (ticker, ts) keys: 0
Rows after removing exact duplicates: 368251
Missing trading days: 0
AAPL is missing 0 trading days between 1980-12-12 and 2024-10-18
AMGN is missing 0 trading days between 1983-06-17 and 2024-10-18
AMZN is missing 0 trading days between 1997-05-15 and 2024-12-06
AXP is missing 0 trading days between 1972-04-03 and 2024-10-18
BA is missing 0 trading days between 1962-01-02 and 2024-10-18
CAT is missing 0 trading days between 1962-01-02 and 2024-10-18
CRM is missing 0 trading days between 2004-06-23 and 2024-10-18
CSCO is missing 0 trading days between 1990-02-16 and 2024-10-18
CVX is missing 0 trading days between 1962-01-02 and 2024-12-06
DIS is missing 0 trading days between 1962-01-02 and 2024-12-06
GS is missing 0 trading days between 1999-05-04 and 2024-10-18
HD is missing 0 trading days between 1981-09-22 and 2024-10-18
HON is missing 0 trading days between 1962-01-02 and 2024-10-18
IBM is missing 0 trading da

In [16]:
test = df[df["ticker"] == "AAPL"].head(10)
test_gap = test.drop(test.index[4])
print("Removed date:", test["ts"].iloc[4])
find_missing_days(test_gap, test)

Removed date: 1980-12-18


,ticker,ts
4,AAPL,1980-12-18


In [17]:
def quality_report(data, missing):
    data = data.copy()
    data["ret"] = data.groupby("ticker")["close"].pct_change()
    data["ret_mean"] = data.groupby("ticker")["ret"].transform("mean")
    data["ret_std"] = data.groupby("ticker")["ret"].transform("std")

    factors = ref[["ticker", "ts"]].copy()
    factors["ref_factor"] = ref["adj_close"] / ref["close"]
    data = data.merge(factors, on=["ticker", "ts"], how="left")

    issues = []

    def add(mask, issue_type, description, severity):
        found = data.loc[mask, ["ticker", "ts"]].copy()
        found["issue_type"] = issue_type
        found["description"] = description
        found["severity"] = severity
        issues.append(found)

    prices = data[["open", "high", "low", "close"]]
    add(prices.isna().any(axis=1) | data["volume"].isna(), "missing_value", "missing price or volume", "high")
    add(data["adj_close"].isna(), "missing_adj_close", "adj_close is missing", "medium")
    add((prices <= 0).any(axis=1) | (data["adj_close"] <= 0), "nonpositive_price", "price <= 0", "high")
    add(np.isinf(prices).any(axis=1) | np.isinf(data["adj_close"]), "non_finite_price", "infinite price", "high")
    add(data["high"] < data["close"], "high_below_close", "high < close", "high")
    add(data["low"] > data["close"], "low_above_close", "low > close", "high")
    add((data["open"] < data["low"]) | (data["open"] > data["high"]), "open_outside_range", "open not between low and high", "high")
    add(abs(data["adj_close"] / data["close"] - data["ref_factor"]) > 1e-4, "adj_close_mismatch", "adj_close not consistent with close and reference factor", "high")
    add(data["volume"] == 0, "zero_volume", "volume is 0, possible stale data", "low")
    add(abs(data["ret"] - data["ret_mean"]) > 3 * data["ret_std"], "return_outlier", "return more than 3 std from the ticker mean", "medium")

    gaps = missing.copy()
    gaps["issue_type"] = "missing_trading_day"
    gaps["description"] = "date in the reference calendar but not in the data"
    gaps["severity"] = "medium"
    issues.append(gaps)

    report = pd.concat(issues, ignore_index=True)
    report = report.rename(columns={"ts": "issue_date"})
    return report[["ticker", "issue_type", "issue_date", "description", "severity"]]

report_before = quality_report(df, missing_days)
print(report_before["issue_type"].value_counts())
report_before.head(10)

issue_type
missing_adj_close     18374
adj_close_mismatch    10417
return_outlier         5056
zero_volume              75
nonpositive_price         5
open_outside_range        5
Name: count, dtype: int64


,ticker,issue_type,issue_date,description,severity
0,AAPL,missing_adj_close,1981-01-08,adj_close is missing,medium
1,AAPL,missing_adj_close,1981-01-13,adj_close is missing,medium
2,AAPL,missing_adj_close,1981-02-12,adj_close is missing,medium
3,AAPL,missing_adj_close,1981-04-16,adj_close is missing,medium
4,AAPL,missing_adj_close,1981-05-19,adj_close is missing,medium
5,AAPL,missing_adj_close,1981-06-04,adj_close is missing,medium
6,AAPL,missing_adj_close,1981-07-14,adj_close is missing,medium
7,AAPL,missing_adj_close,1981-07-20,adj_close is missing,medium
8,AAPL,missing_adj_close,1981-08-26,adj_close is missing,medium
9,AAPL,missing_adj_close,1981-10-23,adj_close is missing,medium


In [18]:
clean = df.copy()

all_missing = clean[["open", "high", "low", "close"]].isna().all(axis=1)
print("Rows with all prices missing:", all_missing.sum())
clean = clean[~all_missing]

bad_open = clean["open"] <= 0
print("Rows with open <= 0:", bad_open.sum())
print(clean[bad_open][["ticker", "ts", "open", "high", "low", "close"]])
clean["open_repaired"] = bad_open
clean.loc[bad_open, "open"] = clean.loc[bad_open, "close"]

Rows with all prices missing: 0
Rows with open <= 0: 5
       ticker          ts  open     high      low    close
87103     CVX  1962-01-02   0.0  3.30000  3.24000  3.30000
167733    JNJ  1962-01-02   0.0  0.22338  0.22222  0.22338
213412    MCD  1966-07-05   0.0  0.27366  0.26749  0.26955
243893    MRK  1962-01-02   0.0  0.39703  0.39317  0.39537
287040     PG  1962-01-02   0.0  1.44000  1.42000  1.43000


In [19]:
clean["prev_close"] = clean.groupby("ticker")["close"].shift(1)
clean["next_close"] = clean.groupby("ticker")["close"].shift(-1)
spike = (clean["close"] > 1.8 * clean["prev_close"]) & (abs(clean["next_close"] / clean["prev_close"] - 1) < 0.05)
print("One-day spikes:", spike.sum())
print(clean[spike][["ticker", "ts", "prev_close", "close", "next_close"]])

for col in ["open", "high", "low", "close"]:
    before = clean.groupby("ticker")[col].shift(1)
    after = clean.groupby("ticker")[col].shift(-1)
    clean.loc[spike, col] = (before[spike] + after[spike]) / 2
clean["spike_repaired"] = spike

One-day spikes: 3
       ticker          ts  prev_close  close  next_close
318332    TRV  1976-10-28        4.61   9.22        4.69
318873    TRV  1978-12-20        4.02   8.03        4.09
319538    TRV  1981-08-10        6.03  12.06        6.03


In [20]:
factors = ref[["ticker", "ts"]].copy()
factors["ref_factor"] = ref["adj_close"] / ref["close"]
clean = clean.merge(factors, on=["ticker", "ts"], how="left")

clean["prev_factor"] = clean.groupby("ticker")["ref_factor"].shift(1)
clean.loc[clean["spike_repaired"], "ref_factor"] = clean.loc[clean["spike_repaired"], "prev_factor"]

bad_adj = clean["adj_close"].isna() | (abs(clean["adj_close"] / clean["close"] - clean["ref_factor"]) > 1e-4)
print("adj_close to rebuild:", bad_adj.sum())
clean["adj_close_repaired"] = bad_adj
clean.loc[bad_adj, "adj_close"] = clean.loc[bad_adj, "close"] * clean.loc[bad_adj, "ref_factor"]

clean["zero_volume"] = clean["volume"] == 0

still_bad = clean[["open", "high", "low", "close", "adj_close"]].isna().any(axis=1)
quarantine = clean[still_bad]
clean = clean[~still_bad]
clean = clean.drop(columns=["prev_close", "next_close", "ref_factor", "prev_factor"])
print("Quarantined rows:", len(quarantine))

adj_close to rebuild: 28793
Quarantined rows: 0


In [21]:
prices = clean[["open", "high", "low", "close", "adj_close"]]
print("no missing prices:", prices.notna().all().all())
print("all prices finite:", np.isfinite(prices).all().all())
print("all prices positive:", (prices > 0).all().all())
print("unique (ticker, ts):", not clean.duplicated(["ticker", "ts"]).any())
print("volume ok:", (clean["volume"] >= 0).all())
print("low <= open <= high:", ((clean["low"] <= clean["open"]) & (clean["open"] <= clean["high"])).all())
print("low <= close <= high:", ((clean["low"] <= clean["close"]) & (clean["close"] <= clean["high"])).all())

report_after = quality_report(clean, find_missing_days(clean, ref))
print("\nIssues after cleaning:")
print(report_after["issue_type"].value_counts())

print("\nRaw rows:", len(raw))
print("Exact duplicates removed:", exact_dup_mask.sum())
print("Quarantined:", len(quarantine))
print("Retained:", len(clean))

no missing prices: True
all prices finite: True
all prices positive: True
unique (ticker, ts): True
volume ok: True
low <= open <= high: True
low <= close <= high: True

Issues after cleaning:
issue_type
return_outlier        5175
zero_volume             75
adj_close_mismatch       3
Name: count, dtype: int64

Raw rows: 397711
Exact duplicates removed: 29460
Quarantined: 0
Retained: 368251


In [22]:
cols = ["open", "high", "low", "close", "adj_close", "volume"]
print("Before:")
print(raw[cols].describe())
print("\nAfter:")
print(clean[cols].describe())

Before:
                open           high            low          close  \
count  397711.000000  397711.000000  397711.000000  397711.000000   
mean       46.285479      46.747660      45.816279      46.292903   
std        67.132594      67.778210      66.477178      67.141879   
min         0.000000       0.005208       0.004801       0.005208   
25%         4.209938       4.240000       4.160000       4.200000   
50%        19.850000      20.060000      19.630000      19.860000   
75%        57.530000      58.130000      56.950000      57.530000   
max       604.260000     608.630000     597.880000     605.400000   

           adj_close        volume  
count  377856.000000  3.977110e+05  
mean       51.652250  3.097773e+07  
std       194.584045  1.266237e+08  
min         0.002945  0.000000e+00  
25%         1.390000  1.730700e+06  
50%        11.960000  4.648200e+06  
75%        43.770000  1.102380e+07  
max     10442.510783  9.276606e+09  

After:
                open         

In [23]:
clean.to_sql("ohlc_hw_cleaned", conn, if_exists="replace", index=False)
pd.read_sql_query("SELECT COUNT(*) FROM ohlc_hw_cleaned", conn)

,COUNT(*)
0,368251


In [24]:
#I built those data casually asking to AI to generate a valid test to test the following code here#
example = pd.DataFrame({
    "ticker": ["TEST"] * 7,
    "ts": ["2024-01-02", "2024-01-03", "2024-01-04", "2024-01-05", "2024-01-08", "2024-01-09", "2024-01-09"],
    "open": [10.0, 10.5, 11.0, 11.2, np.nan, 11.5, 11.5],
    "high": [10.8, 11.0, 11.1, 11.6, np.nan, 11.9, 11.9],
    "low": [9.9, 10.2, 10.7, 11.0, np.nan, 11.3, 11.3],
    "close": [10.5, np.nan, 11.3, 11.4, np.nan, 11.6, 11.8],
    "adj_close": [10.5, np.nan, 11.3, 11.4, np.nan, 11.6, 11.8],
    "volume": [1000, 1200, 900, np.nan, 800, 950, 950],
})
example

,ticker,ts,open,high,low,close,adj_close,volume
0,TEST,2024-01-02,10.0,10.8,9.9,10.5,10.5,1000.0
1,TEST,2024-01-03,10.5,11.0,10.2,NaN,NaN,1200.0
2,TEST,2024-01-04,11.0,11.1,10.7,11.3,11.3,900.0
3,TEST,2024-01-05,11.2,11.6,11.0,11.4,11.4,NaN
4,TEST,2024-01-08,NaN,NaN,NaN,NaN,NaN,800.0
5,TEST,2024-01-09,11.5,11.9,11.3,11.6,11.6,950.0
6,TEST,2024-01-09,11.5,11.9,11.3,11.8,11.8,950.0


In [25]:
print("Conflicting (ticker, ts) rows:", example.duplicated(["ticker", "ts"], keep=False).sum())
quality_report(example, missing_days.iloc[0:0])[["issue_type", "issue_date"]]

Conflicting (ticker, ts) rows: 2


C:\Users\Leonardo Martini\AppData\Local\Temp\ipykernel_14120\3224485697.py:3: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  data["ret"] = data.groupby("ticker")["close"].pct_change()


,issue_type,issue_date
0,missing_value,2024-01-03
1,missing_value,2024-01-05
2,missing_value,2024-01-08
3,missing_adj_close,2024-01-03
4,missing_adj_close,2024-01-08
5,high_below_close,2024-01-04


In [26]:
def repair(data):
    data = data[~data.duplicated(["ticker", "ts"], keep=False)].copy()
    data = data[~data[["open", "high", "low", "close"]].isna().all(axis=1)].copy()

    data["volume_filled"] = data["volume"].isna()
    data["volume"] = data["volume"].fillna(0)

    bad = (data["high"] < data["close"]) | (data["low"] > data["close"])
    data.loc[bad, "high"] = data.loc[bad, ["open", "close"]].max(axis=1)

    data["price_filled"] = data["close"].isna()
    data[["open", "high", "low", "close", "adj_close"]] = data.groupby("ticker")[["open", "high", "low", "close", "adj_close"]].ffill()
    data["close"] = data["close"].clip(data["low"], data["high"])
    return data

fixed = repair(example)
print("low <= close <= high:", ((fixed["low"] <= fixed["close"]) & (fixed["close"] <= fixed["high"])).all())
fixed

low <= close <= high: True


,ticker,ts,open,high,low,close,adj_close,volume,volume_filled,price_filled
0,TEST,2024-01-02,10.0,10.8,9.9,10.5,10.5,1000.0,False,False
1,TEST,2024-01-03,10.5,11.0,10.2,10.5,10.5,1200.0,False,True
2,TEST,2024-01-04,11.0,11.3,10.7,11.3,11.3,900.0,False,False
3,TEST,2024-01-05,11.2,11.6,11.0,11.4,11.4,0.0,True,False


In [28]:
#the cleaning choices made
 #Some tickers were written inside HTML tags, so I removed the HTML before grouping the data. I kept the original label in `ticker_raw`.
 #I removed 29,460 rows that were exact copies. I didn't find two different rows with the same ticker and date. In the test example I delete both rows when this happens, because I can't know which one is correct.
 #For missing days I used the dates in the `ohlc` table as the calendar for each ticker. There are no missing days in the data, and the test with one removed date works.
 #In 5 rows the open price is 0. They are all on the first day of the stock history, so there is no previous close to use. I used the close of the same day instead.
 #On 3 days TRV's price doubles and then goes back to normal the next day. This is clearly a data error, so I replaced these prices with the average of the day before and the day after.
 #adj_close was missing in 18,374 rows and wrong in 10,417 rows (4 to 24 times too big). The raw close is the same as in the `ohlc` table, so I calculated adj_close again as close * (adj_close / close in `ohlc`) for the same day. On the 3 TRV error days the `ohlc` table is also wrong, so I used the factor of the day before. The factor is the same before and after these days, so there is no split or dividend there.
 #75 days have volume 0. I kept them with a flag, because it can be a real day with no trading.
 #Returns bigger than 3 standard deviations are kept with a flag.#